# Semantic Model Similarity - Results

Interactive app for the semantic model similarity analysis. It reads the scored results from the lakehouse (written by the **semantic_model_similarity** notebook) and renders them below. Run the notebook top to bottom and explore - there is no code to edit.

In [ ]:
def render_results():
    """Load the latest results from the lakehouse and render the interactive app."""
    import itertools
    import re
    from collections import defaultdict
    
    import numpy as np
    import pandas as pd
    
    def load_delta(table_name):
        df = spark.read.format("delta").load('Tables/' + table_name)
        return df.toPandas()
    
    
    models_df = load_delta("semantic_models")
    tables_df = load_delta("semantic_model_tables")
    columns_df = load_delta("semantic_model_columns")
    relationships_df = load_delta("semantic_model_relationships")
    measures_df = load_delta("semantic_model_measures")
    datasources_df = load_delta("semantic_model_datasources")
    
    if models_df.empty:
        raise ValueError(
            "No rows in the semantic_models table of the attached lakehouse. "
            "Run the TOM catalog notebook first."
        )
    
    print(f"Models: {len(models_df)}")
    print(
        f"Tables: {len(tables_df)} | Columns: {len(columns_df)} | "
        f"Measures: {len(measures_df)} | Relationships: {len(relationships_df)} | "
        f"Datasources: {len(datasources_df)}"
    )
    
    # Scored outputs + run metadata written by the semantic_model_similarity notebook.
    pairs_df = load_delta("semantic_model_similarity_pairs")
    clusters_df = load_delta("semantic_model_duplicate_clusters")
    signatures_df = load_delta("semantic_model_signatures")
    try:
        run_meta_df = load_delta("semantic_model_similarity_run")
    except Exception:
        run_meta_df = pd.DataFrame()  # optional; falls back to default thresholds
    
    def norm(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return re.sub(r"\s+", " ", str(value)).strip().casefold()
    
    
    def norm_dax(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        text = str(value)
        text = re.sub(r"/\*.*?\*/", " ", text, flags=re.S)  # block comments
        text = re.sub(r"//.*", " ", text)  # line comments
        text = re.sub(r"\s+", " ", text)
        return text.strip().casefold()
    
    
    def model_label(sig):
        return f"{sig['workspace_name']} / {sig['model_name']}"
    
    
    signatures = {}
    for _, row in models_df.iterrows():
        model_id = str(row["model_id"])
        signatures[model_id] = {
            "model_id": model_id,
            "workspace_id": str(row.get("workspace_id", "")),
            "workspace_name": str(row.get("workspace_name", "")),
            "model_name": str(row.get("model_name", "")),
            "tables": set(),
            "columns": set(),
            "measure_names": set(),
            "measure_definitions": set(),
            "relationships": set(),
            "datasources": set(),
            "dax_docs": [],
        }
    
    
    def sig_for(model_id):
        return signatures.get(str(model_id))
    
    
    for _, row in tables_df.iterrows():
        sig = sig_for(row["model_id"])
        if sig is not None:
            sig["tables"].add(norm(row["table_name"]))
    
    for _, row in columns_df.iterrows():
        sig = sig_for(row["model_id"])
        if sig is not None:
            sig["columns"].add(f"{norm(row['table_name'])}.{norm(row['column_name'])}")
    
    for _, row in measures_df.iterrows():
        sig = sig_for(row["model_id"])
        if sig is not None:
            measure_name = norm(row["measure_name"])
            measure_dax = norm_dax(row.get("expression"))
            sig["measure_names"].add(measure_name)
            # Name + DAX key so containment only credits measures whose logic also matches.
            sig["measure_definitions"].add(f"{measure_name} :: {measure_dax}")
            sig["dax_docs"].append(f"{measure_name} {measure_dax}".strip())
    
    for _, row in relationships_df.iterrows():
        sig = sig_for(row["model_id"])
        if sig is not None:
            key = (
                f"{norm(row['from_table'])}.{norm(row['from_column'])}"
                f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
            )
            sig["relationships"].add(key)
    
    for _, row in datasources_df.iterrows():
        sig = sig_for(row["model_id"])
        if sig is not None:
            conn = row.get("connection_string") or row.get("connection_details") or row.get("datasource_name")
            conn_norm = norm(conn)
            if conn_norm:
                sig["datasources"].add(conn_norm)
    
    # Build the embedding document per model, with a structural fallback when no measures exist.
    for sig in signatures.values():
        parts = list(sig["dax_docs"])
        if not parts:
            parts = sorted(sig["tables"]) + sorted(sig["columns"])
        sig["doc"] = " \n ".join(parts) if parts else (sig["model_name"] or sig["model_id"])
    
    model_ids = list(signatures.keys())
    print(f"Built signatures for {len(model_ids)} models.")
    
    # Default thresholds and summary counts come from the compute run's metadata table.
    if not run_meta_df.empty:
        _m = run_meta_df.iloc[0]
        DUPLICATE_THRESHOLD = float(_m["duplicate_threshold"])
        SIMILAR_THRESHOLD = float(_m["similar_threshold"])
        CONTAINMENT_THRESHOLD = float(_m["containment_threshold"])
    else:
        DUPLICATE_THRESHOLD, SIMILAR_THRESHOLD, CONTAINMENT_THRESHOLD = 0.95, 0.70, 0.95
    
    duplicate_count = int((pairs_df["tier"] == "duplicate").sum()) if not pairs_df.empty else 0
    similar_count = int((pairs_df["tier"] == "similar").sum()) if not pairs_df.empty else 0
    containment_count = int((pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD).sum()) if not pairs_df.empty else 0
    cluster_number = int(clusters_df["cluster_id"].nunique()) if not clusters_df.empty else 0
    
    # Interactive results app: a single navigable in-notebook experience rendered via displayHTML.
    # All data is embedded; navigation, search, drill-down, structure-diff and theme run client-side.
    import json
    import hashlib
    from datetime import datetime, timezone
    
    # Shared DAX pool so identical measure expressions (common across near-duplicate models) embed once.
    _dax_pool = []
    _dax_index = {}
    
    
    def _dax_id(text):
        if not text:
            return -1
        idx = _dax_index.get(text)
        if idx is None:
            idx = len(_dax_pool)
            _dax_pool.append(text)
            _dax_index[text] = idx
        return idx
    
    
    def _dax_hash(norm_text):
        return hashlib.md5(norm_text.encode("utf-8")).hexdigest()[:12] if norm_text else ""
    
    
    def _round(value):
        try:
            return round(float(value), 4)
        except (TypeError, ValueError):
            return None
    
    
    def _disp(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return str(value)
    
    
    total_models = int(len(signatures_df)) if not signatures_df.empty else len(signatures)
    
    if not pairs_df.empty:
        _flagged = pairs_df[pairs_df["tier"].isin(["duplicate", "similar"])].sort_values("composite_score", ascending=False)
        _contained = pairs_df[pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD].sort_values("containment_score", ascending=False)
    else:
        _flagged = pairs_df
        _contained = pairs_df
    
    
    def _pair_obj(row):
        return {
            "tier": row["tier"],
            "composite": _round(row["composite_score"]),
            "containment": _round(row["containment_score"]),
            "relationship": row.get("containment_relationship", ""),
            "idA": str(row["model_id_a"]), "idB": str(row["model_id_b"]),
            "modelA": str(row["model_a"]), "workspaceA": str(row["workspace_a"]),
            "modelB": str(row["model_b"]), "workspaceB": str(row["workspace_b"]),
            "aInB": _round(row["model_a_in_model_b"]), "bInA": _round(row["model_b_in_model_a"]),
            "crossWorkspace": bool(row.get("cross_workspace", False)),
            "sameName": bool(row.get("same_model_name", False)),
            "jaccard": {
                "tables": _round(row["jaccard_tables"]), "columns": _round(row["jaccard_columns"]),
                "measures": _round(row["jaccard_measure_names"]), "relationships": _round(row["jaccard_relationships"]),
                "datasources": _round(row["jaccard_datasources"]),
            },
            "daxCosine": _round(row["dax_embedding_cosine"]),
        }
    
    
    _pairs_payload = [_pair_obj(r) for _, r in _flagged.iterrows()]
    _containment_payload = [_pair_obj(r) for _, r in _contained.iterrows()]
    _all_pairs_payload = [_pair_obj(r) for _, r in pairs_df.iterrows()]
    
    _clusters_payload = []
    if not clusters_df.empty:
        for _cid, _grp in clusters_df.sort_values(["cluster_id", "model"]).groupby("cluster_id"):
            _clusters_payload.append({
                "id": int(_cid),
                "size": int(_grp["cluster_size"].iloc[0]),
                "members": [{"id": str(m["model_id"]), "model": str(m["model_name"]), "workspace": str(m["workspace_name"])} for _, m in _grp.iterrows()],
            })
    
    _matrix_payload = {"labels": [], "workspaces": [], "z": []}
    if not _flagged.empty:
        _ids = sorted(
            set(_flagged["model_id_a"]) | set(_flagged["model_id_b"]),
            key=lambda mid: (signatures[mid]["model_name"], signatures[mid]["workspace_name"]),
        )
        _pos = {mid: index for index, mid in enumerate(_ids)}
        _n = len(_ids)
        _z = np.eye(_n)
        for _, row in _flagged.iterrows():
            i, j = _pos[row["model_id_a"]], _pos[row["model_id_b"]]
            _z[i, j] = _z[j, i] = float(row["composite_score"])
        _matrix_payload = {
            "labels": [signatures[mid]["model_name"] for mid in _ids],
            "workspaces": [signatures[mid]["workspace_name"] for mid in _ids],
            "z": [[_round(v) for v in r] for r in _z.tolist()],
        }
    
    # Per-model object inventories for the Compare / structure-diff tab (only models in a pair/cluster).
    _involved = set()
    if not pairs_df.empty:
        _involved |= set(pairs_df["model_id_a"]) | set(pairs_df["model_id_b"])
    if not clusters_df.empty:
        _involved |= set(clusters_df["model_id"])
    _involved = {str(x) for x in _involved}
    
    _inv = {mid: {"tables": {}, "columns": {}, "measures": {}, "relationships": {}, "datasources": {}} for mid in _involved}
    
    for _, r in tables_df.iterrows():
        mid = str(r["model_id"])
        if mid in _inv:
            k = norm(r["table_name"])
            if k and k not in _inv[mid]["tables"]:
                _inv[mid]["tables"][k] = _disp(r["table_name"])
    for _, r in columns_df.iterrows():
        mid = str(r["model_id"])
        if mid in _inv:
            tk, ck = norm(r["table_name"]), norm(r["column_name"])
            key = tk + "." + ck
            if ck and key not in _inv[mid]["columns"]:
                _inv[mid]["columns"][key] = {"table": _disp(r["table_name"]), "tableKey": tk, "name": _disp(r["column_name"])}
    for _, r in measures_df.iterrows():
        mid = str(r["model_id"])
        if mid in _inv:
            nk = norm(r["measure_name"])
            if nk and nk not in _inv[mid]["measures"]:
                _inv[mid]["measures"][nk] = {"name": _disp(r["measure_name"]), "daxId": _dax_id(_disp(r.get("expression"))), "daxHash": _dax_hash(norm_dax(r.get("expression")))}
    for _, r in relationships_df.iterrows():
        mid = str(r["model_id"])
        if mid in _inv:
            key = f"{norm(r['from_table'])}.{norm(r['from_column'])}->{norm(r['to_table'])}.{norm(r['to_column'])}"
            if key not in _inv[mid]["relationships"]:
                _inv[mid]["relationships"][key] = {"from": f"{_disp(r['from_table'])}[{_disp(r['from_column'])}]", "to": f"{_disp(r['to_table'])}[{_disp(r['to_column'])}]"}
    for _, r in datasources_df.iterrows():
        mid = str(r["model_id"])
        if mid in _inv:
            conn = r.get("connection_string") or r.get("connection_details") or r.get("datasource_name")
            k = norm(conn)
            if k and k not in _inv[mid]["datasources"]:
                _inv[mid]["datasources"][k] = _disp(conn)
    
    _models_payload = {}
    for mid in _involved:
        inv = _inv[mid]
        sig = signatures.get(mid, {})
        _models_payload[mid] = {
            "name": _disp(sig.get("model_name", "")),
            "workspace": _disp(sig.get("workspace_name", "")),
            "tables": [{"key": k, "name": v} for k, v in inv["tables"].items()],
            "columns": [{"key": k, "table": v["table"], "tableKey": v["tableKey"], "name": v["name"]} for k, v in inv["columns"].items()],
            "measures": [{"key": k, "name": v["name"], "daxId": v["daxId"], "daxHash": v["daxHash"]} for k, v in inv["measures"].items()],
            "relationships": [{"key": k, "from": v["from"], "to": v["to"]} for k, v in inv["relationships"].items()],
            "datasources": [{"key": k, "name": v} for k, v in inv["datasources"].items()],
        }
    
    _model_list = sorted(
        [{"id": mid, "name": _models_payload[mid]["name"], "workspace": _models_payload[mid]["workspace"]} for mid in _involved],
        key=lambda d: (d["name"].casefold(), d["workspace"].casefold()),
    )
    _default_a = _default_b = ""
    if _pairs_payload:
        _default_a, _default_b = _pairs_payload[0]["idA"], _pairs_payload[0]["idB"]
    elif len(_model_list) >= 2:
        _default_a, _default_b = _model_list[0]["id"], _model_list[1]["id"]
    
    _app_data = {
        "generatedAt": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
        "summary": {
            "models": int(total_models),
            "duplicatePairs": int(duplicate_count),
            "similarPairs": int(similar_count),
            "containmentCandidates": int(containment_count),
            "clusters": int(cluster_number),
        },
        "thresholds": {
            "duplicate": _round(DUPLICATE_THRESHOLD),
            "similar": _round(SIMILAR_THRESHOLD),
            "containment": _round(CONTAINMENT_THRESHOLD),
        },
        "pairs": _pairs_payload,
        "allPairs": _all_pairs_payload,
        "containment": _containment_payload,
        "clusters": _clusters_payload,
        "matrix": _matrix_payload,
        "models": _models_payload,
        "daxPool": _dax_pool,
        "modelList": _model_list,
        "defaultCompare": {"a": _default_a, "b": _default_b},
    }
    
    _APP_TEMPLATE = r"""<style>
    #sms-app{--bg:#f5f5f7;--card:#ffffff;--text:#1d1d1f;--soft:#3a3a3c;--muted:#6e6e73;--faint:#8a8a8e;--hair:#f0f0f2;--track:#ececee;--border:#e0e0e3;--vs:#c7c7cc;--accent:#0071e3;--accent2:#0058c9;--tabbar:#e9e9ec;--tcount:rgba(0,0,0,.06);--shadow:0 1px 3px rgba(0,0,0,.06);--dupbg:#ffe9e3;--dupfg:#c9330a;--simbg:#fff2df;--simfg:#96590a;--chipbg:#eef1f6;--chipfg:#5b5b60;--xwsbg:#efe9fc;--xwsfg:#6b3fd4;--badgebg:#eef4ff;--mini:#8e8e93;--focus:rgba(0,113,227,.14);--mbar:linear-gradient(90deg,hsl(211,88%,94%),hsl(211,88%,36%));--dAbg:#e6f0ff;--dAfg:#0058c9;--dBbg:#fff1e0;--dBfg:#a5590a;--dCbg:#f1e9ff;--dCfg:#6b3fd4;--addbg:#e6f7ec;--addfg:#1a7f37;--delbg:#ffeef0;--delfg:#c9330a;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif;color:var(--text);background:var(--bg);border-radius:20px;max-width:1120px;margin:8px auto;overflow:hidden;-webkit-font-smoothing:antialiased;}
    #sms-app[data-theme="dark"]{--bg:#1c1c1e;--card:#2c2c2e;--text:#f5f5f7;--soft:#d1d1d6;--muted:#98989d;--faint:#8e8e93;--hair:#3a3a3c;--track:#3a3a3c;--border:#48484a;--vs:#5a5a5e;--accent:#0a84ff;--accent2:#409cff;--tabbar:#000000;--tcount:rgba(255,255,255,.12);--shadow:0 1px 3px rgba(0,0,0,.45);--dupbg:rgba(255,69,58,.22);--dupfg:#ff6b5e;--simbg:rgba(255,159,10,.22);--simfg:#ffb340;--chipbg:#3a3a3c;--chipfg:#c7c7cc;--xwsbg:rgba(120,90,220,.30);--xwsfg:#cbb8ff;--badgebg:rgba(10,132,255,.20);--focus:rgba(10,132,255,.30);--mbar:linear-gradient(90deg,hsl(211,55%,26%),hsl(211,90%,62%));--dAbg:rgba(10,132,255,.18);--dAfg:#7db8ff;--dBbg:rgba(255,159,10,.18);--dBfg:#ffca6b;--dCbg:rgba(150,110,240,.22);--dCfg:#cbb8ff;--addbg:rgba(48,209,88,.16);--addfg:#63e6a0;--delbg:rgba(255,69,58,.18);--delfg:#ff8a80;}
    #sms-app *{box-sizing:border-box;}
    #sms-app .apphead{display:flex;align-items:flex-start;justify-content:space-between;gap:16px;padding:24px 28px 16px;}
    #sms-app .eyebrow{font-size:12px;font-weight:600;letter-spacing:.07em;text-transform:uppercase;color:var(--faint);}
    #sms-app .apptitle{font-size:22px;font-weight:600;letter-spacing:-.01em;margin-top:3px;}
    #sms-app .hright{display:flex;flex-direction:column;align-items:flex-end;gap:9px;}
    #sms-app .themeswitch{display:flex;gap:3px;padding:3px;background:var(--tabbar);border-radius:9px;}
    #sms-app .tsw{border:0;background:transparent;color:var(--muted);font-size:13px;line-height:1;padding:5px 10px;border-radius:7px;cursor:pointer;}
    #sms-app .tsw.on{background:var(--card);color:var(--accent);box-shadow:var(--shadow);}
    #sms-app .gen{font-size:12px;color:var(--faint);}
    #sms-app .tabs{display:flex;gap:4px;padding:4px;margin:0 28px;background:var(--tabbar);border-radius:12px;overflow-x:auto;}
    #sms-app .tab{flex:0 0 auto;border:0;background:transparent;color:var(--muted);font:inherit;font-size:13px;font-weight:600;padding:8px 16px;border-radius:9px;cursor:pointer;display:flex;align-items:center;gap:7px;white-space:nowrap;}
    #sms-app .tab.active{background:var(--card);color:var(--accent);box-shadow:var(--shadow);}
    #sms-app .tcount{font-size:11px;font-weight:600;background:var(--tcount);color:var(--muted);border-radius:999px;padding:1px 7px;}
    #sms-app .tab.active .tcount{background:var(--badgebg);color:var(--accent);}
    #sms-app #sms-view{padding:22px 28px 26px;}
    #sms-app .head2{font-size:19px;font-weight:600;letter-spacing:-.01em;line-height:1.32;margin-bottom:18px;}
    #sms-app .kpis{display:flex;flex-wrap:wrap;gap:12px;}
    #sms-app .kpi{flex:1 1 130px;background:var(--card);border-radius:14px;padding:16px 18px;box-shadow:var(--shadow);}
    #sms-app .kpi .n{font-size:30px;font-weight:600;letter-spacing:-.02em;line-height:1;}
    #sms-app .kpi .l{font-size:12px;color:var(--muted);margin-top:6px;}
    #sms-app .kpi.accent .n{color:var(--accent);}
    #sms-app .kpi.warn .n{color:var(--dupfg);}
    #sms-app .secrow{display:flex;align-items:center;justify-content:space-between;margin:24px 0 12px;}
    #sms-app .sec{font-size:16px;font-weight:600;margin:0;}
    #sms-app .btnlink{border:0;background:transparent;color:var(--accent);font:inherit;font-size:13px;font-weight:600;cursor:pointer;padding:0;}
    #sms-app .toolbar{display:flex;gap:10px;align-items:center;margin-bottom:14px;flex-wrap:wrap;}
    #sms-app .search{flex:1 1 240px;min-width:180px;border:1px solid var(--border);background:var(--card);border-radius:10px;padding:9px 13px;font:inherit;font-size:13px;color:var(--text);}
    #sms-app .search::placeholder{color:var(--faint);}
    #sms-app .search:focus{outline:none;border-color:var(--accent);box-shadow:0 0 0 3px var(--focus);}
    #sms-app .chips{display:flex;gap:6px;}
    #sms-app .fchip{border:1px solid var(--border);background:var(--card);color:var(--muted);font:inherit;font-size:12.5px;font-weight:600;padding:8px 13px;border-radius:999px;cursor:pointer;}
    #sms-app .fchip.on{background:var(--accent);border-color:var(--accent);color:#fff;}
    #sms-app .plist,#sms-app .clist{display:flex;flex-direction:column;gap:10px;}
    #sms-app .pcard{background:var(--card);border-radius:14px;box-shadow:var(--shadow);overflow:hidden;}
    #sms-app .phead{display:flex;align-items:center;gap:16px;padding:14px 16px;cursor:pointer;}
    #sms-app .pinfo{flex:1 1 auto;min-width:0;}
    #sms-app .ptop{display:flex;align-items:center;gap:6px;margin-bottom:7px;}
    #sms-app .pnames{font-size:14px;line-height:1.5;}
    #sms-app .mname{font-weight:600;}
    #sms-app .wname{font-size:12px;color:var(--faint);}
    #sms-app .vs{color:var(--vs);margin:0 7px;}
    #sms-app .pscore{flex:0 0 170px;}
    #sms-app .track{height:7px;background:var(--track);border-radius:6px;overflow:hidden;}
    #sms-app .fill{height:100%;border-radius:6px;background:var(--accent);}
    #sms-app .fill.cov{background:var(--accent2);}
    #sms-app .fill.mini{background:var(--mini);}
    #sms-app .scoreval{font-size:11.5px;color:var(--muted);margin-top:4px;}
    #sms-app .chev{flex:0 0 auto;color:var(--vs);font-size:20px;line-height:1;transition:transform .15s ease;}
    #sms-app .pcard.open .chev{transform:rotate(90deg);}
    #sms-app .pdetail{display:none;padding:0 16px 16px;}
    #sms-app .pcard.open .pdetail{display:block;}
    #sms-app .verdict{font-size:13.5px;color:var(--soft);border-top:1px solid var(--hair);padding-top:12px;margin-bottom:12px;}
    #sms-app .evid{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));gap:10px 16px;}
    #sms-app .ev{font-size:12px;}
    #sms-app .evl{color:var(--faint);margin-bottom:5px;}
    #sms-app .ev .track{height:5px;}
    #sms-app .evv{font-size:11.5px;color:var(--soft);margin-top:3px;font-weight:600;}
    #sms-app .pdact{margin-top:12px;}
    #sms-app .pill{display:inline-block;font-size:11px;font-weight:600;padding:2px 9px;border-radius:999px;}
    #sms-app .pill.dup{background:var(--dupbg);color:var(--dupfg);}
    #sms-app .pill.sim{background:var(--simbg);color:var(--simfg);}
    #sms-app .chip{display:inline-block;font-size:10.5px;font-weight:500;padding:2px 8px;border-radius:999px;background:var(--chipbg);color:var(--chipfg);}
    #sms-app .chip.xws{background:var(--xwsbg);color:var(--xwsfg);}
    #sms-app .grid2{display:grid;grid-template-columns:repeat(auto-fill,minmax(260px,1fr));gap:12px;}
    #sms-app .card{background:var(--card);border-radius:14px;padding:16px;box-shadow:var(--shadow);}
    #sms-app .crow{display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;}
    #sms-app .badge{font-size:11px;font-weight:600;color:var(--accent);background:var(--badgebg);border-radius:999px;padding:3px 10px;}
    #sms-app .members{list-style:none;margin:0;padding:0;}
    #sms-app .members li{display:flex;justify-content:space-between;align-items:baseline;padding:7px 0;border-top:1px solid var(--hair);}
    #sms-app .members li:first-child{border-top:0;}
    #sms-app .members .mm{font-size:13.5px;font-weight:600;}
    #sms-app .members .w{font-size:12px;color:var(--faint);}
    #sms-app .cbtnrow{margin-top:12px;}
    #sms-app .empty{background:var(--card);border-radius:14px;padding:28px;text-align:center;color:var(--muted);font-size:14px;box-shadow:var(--shadow);}
    #sms-app .mwrap{overflow:auto;background:var(--card);border-radius:14px;padding:16px;box-shadow:var(--shadow);}
    #sms-app .mgrid{display:grid;gap:2px;align-items:center;}
    #sms-app .mcolh{font-size:11px;color:var(--faint);text-align:center;font-weight:600;}
    #sms-app .mrowh{font-size:12px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;padding-right:8px;color:var(--text);}
    #sms-app .mrowh .mnum{display:inline-block;width:20px;color:var(--faint);font-weight:600;}
    #sms-app .mcell{width:30px;height:30px;border-radius:5px;}
    #sms-app .mlegend{display:flex;align-items:center;gap:8px;margin-top:14px;font-size:11.5px;color:var(--faint);}
    #sms-app .mbar{height:8px;width:120px;border-radius:6px;background:var(--mbar);}
    #sms-app .foot{font-size:11.5px;color:var(--faint);padding:0 28px 22px;line-height:1.6;}
    #sms-app .cmpbar{display:flex;flex-wrap:wrap;gap:12px;align-items:center;justify-content:space-between;margin-bottom:16px;}
    #sms-app .cmppick{display:flex;align-items:center;gap:8px;flex:1 1 auto;flex-wrap:wrap;}
    #sms-app .cmpsel{flex:1 1 220px;min-width:150px;border:1px solid var(--border);background:var(--card);color:var(--text);border-radius:10px;padding:8px 11px;font:inherit;font-size:13px;}
    #sms-app .cmpsel:focus{outline:none;border-color:var(--accent);box-shadow:0 0 0 3px var(--focus);}
    #sms-app .swap{border:1px solid var(--border);background:var(--card);color:var(--muted);border-radius:9px;padding:7px 11px;cursor:pointer;font-size:14px;}
    #sms-app .difftoggle{font-size:12.5px;color:var(--muted);display:flex;align-items:center;gap:7px;cursor:pointer;white-space:nowrap;}
    #sms-app .cmphead{display:flex;align-items:center;gap:12px;margin-bottom:16px;font-size:14px;flex-wrap:wrap;}
    #sms-app .cmpm{font-weight:600;display:flex;align-items:center;gap:7px;}
    #sms-app .cmpvs{color:var(--vs);font-weight:400;}
    #sms-app .dot{width:10px;height:10px;border-radius:3px;display:inline-block;}
    #sms-app .dotA{background:var(--dAfg);}
    #sms-app .dotB{background:var(--dBfg);}
    #sms-app .diffsec{background:var(--card);border-radius:14px;box-shadow:var(--shadow);padding:14px 16px;margin-bottom:12px;}
    #sms-app .diffhead{display:flex;align-items:baseline;justify-content:space-between;gap:12px;margin-bottom:8px;flex-wrap:wrap;cursor:pointer;}
    #sms-app .diffhead .chevs{display:inline-block;color:var(--vs);font-size:15px;margin-right:6px;transition:transform .15s ease;transform:rotate(90deg);}
    #sms-app .diffsec.collapsed .diffhead .chevs{transform:rotate(0deg);}
    #sms-app .diffsec.collapsed .diffhead{margin-bottom:0;}
    #sms-app .diffsec.collapsed .diffbody{display:none;}
    #sms-app .dtitle{font-size:15px;font-weight:600;}
    #sms-app .dcounts{font-size:12px;color:var(--muted);}
    #sms-app .dcounts .cA{color:var(--dAfg);font-weight:600;}
    #sms-app .dcounts .cB{color:var(--dBfg);font-weight:600;}
    #sms-app .dcounts .cchg{color:var(--dCfg);font-weight:600;}
    #sms-app .diffrow{display:flex;align-items:center;gap:10px;padding:6px 8px;border-radius:8px;font-size:13px;}
    #sms-app .diffrow.onlyA{background:var(--dAbg);}
    #sms-app .diffrow.onlyB{background:var(--dBbg);}
    #sms-app .diffrow.changed{background:var(--dCbg);}
    #sms-app .stag{flex:0 0 auto;font-size:10px;font-weight:700;min-width:20px;text-align:center;border-radius:5px;padding:2px 5px;color:#fff;}
    #sms-app .stag.both{background:var(--track);color:var(--muted);}
    #sms-app .stag.onlyA{background:var(--dAfg);}
    #sms-app .stag.onlyB{background:var(--dBfg);}
    #sms-app .stag.changed{background:var(--dCfg);}
    #sms-app .dtext{flex:1 1 auto;min-width:0;}
    #sms-app .difftbl .thead{cursor:pointer;}
    #sms-app .difftbl .chev2{display:inline-block;color:var(--vs);font-size:15px;transition:transform .15s ease;}
    #sms-app .difftbl .chev2.hide{visibility:hidden;}
    #sms-app .difftbl.open .chev2{transform:rotate(90deg);}
    #sms-app .difftbl .cols{display:none;padding:2px 0 6px 30px;}
    #sms-app .difftbl.open .cols{display:block;}
    #sms-app .diffcol{display:flex;align-items:center;gap:9px;padding:4px 8px;border-radius:7px;font-size:12.5px;margin-top:2px;}
    #sms-app .diffcol.onlyA{background:var(--dAbg);}
    #sms-app .diffcol.onlyB{background:var(--dBbg);}
    #sms-app .tmeta{font-size:11.5px;color:var(--faint);margin-left:8px;}
    #sms-app .tmeta .cchg{color:var(--dCfg);font-weight:600;}
    #sms-app .mdax{margin-top:6px;font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:12px;border-radius:8px;overflow:hidden;border:1px solid var(--hair);}
    #sms-app .dline{padding:1px 10px;white-space:pre-wrap;word-break:break-word;}
    #sms-app .dline.same{color:var(--muted);}
    #sms-app .dline.add{background:var(--addbg);color:var(--addfg);}
    #sms-app .dline.del{background:var(--delbg);color:var(--delfg);}
    #sms-app .dsplit{display:grid;grid-template-columns:1fr 1fr;gap:8px;}
    #sms-app .dcap{font-size:12px;color:var(--faint);margin:4px 0 6px;}
    #sms-app .diffnone{font-size:12.5px;color:var(--faint);padding:4px 8px;}
    #sms-app .hctrls{display:flex;gap:8px;align-items:center;}
    #sms-app .gear{border:0;background:var(--tabbar);color:var(--muted);font-size:15px;line-height:1;width:30px;height:30px;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;justify-content:center;}
    #sms-app .gear.on{background:var(--card);color:var(--accent);box-shadow:var(--shadow);}
    #sms-app .settings{max-height:0;overflow:hidden;transition:max-height .28s ease;margin:0 28px;}
    #sms-app .settings.open{max-height:420px;}
    #sms-app .setinner{background:var(--card);border-radius:14px;box-shadow:var(--shadow);padding:14px 18px;margin-top:8px;}
    #sms-app .setrow{display:grid;grid-template-columns:210px 1fr 66px;align-items:center;gap:14px;padding:9px 0;}
    #sms-app .setrow + .setrow{border-top:1px solid var(--hair);}
    #sms-app .setlab{font-size:13px;font-weight:600;display:flex;flex-direction:column;gap:2px;}
    #sms-app .sethint{font-size:11px;font-weight:400;color:var(--faint);}
    #sms-app .setrange{width:100%;accent-color:var(--accent);}
    #sms-app .setnum{border:1px solid var(--border);background:var(--bg);color:var(--text);border-radius:8px;padding:6px 8px;font:inherit;font-size:13px;width:100%;}
    #sms-app .setfoot{display:flex;align-items:center;justify-content:space-between;gap:12px;margin-top:10px;padding-top:10px;border-top:1px solid var(--hair);}
    #sms-app .setnote{font-size:11.5px;color:var(--faint);}
    </style>
    <div id="sms-app"></div>
    <script>
    (function(){
      var DATA = __APP_DATA__;
      var root = document.getElementById('sms-app');
      if(!root){ return; }
      var S = DATA.summary, T = DATA.thresholds, MODELS = DATA.models || {}, MLIST = DATA.modelList || [], DAXPOOL = DATA.daxPool || [];
      function daxText(m){ return (m && m.daxId >= 0) ? (DAXPOOL[m.daxId] || '') : ''; }
      var state = { tab:'overview', pairSearch:'', pairTier:'all', contSearch:'', theme:'light',
                    cmpA:(DATA.defaultCompare&&DATA.defaultCompare.a)||'', cmpB:(DATA.defaultCompare&&DATA.defaultCompare.b)||'', cmpDiffOnly:false, cmpCollapsed:{tables:true,measures:true,relationships:true,datasources:true} };
      if(!state.cmpA && MLIST[0]){ state.cmpA = MLIST[0].id; }
      if(!state.cmpB && MLIST[1]){ state.cmpB = MLIST[1].id; }
      try {
        var saved = localStorage.getItem('sms-theme');
        if(saved){ state.theme = saved; }
        else if(window.matchMedia && window.matchMedia('(prefers-color-scheme: dark)').matches){ state.theme = 'dark'; }
      } catch(e){ }
    
      state.thresholds = { duplicate:(T&&T.duplicate!=null)?T.duplicate:0.95, similar:(T&&T.similar!=null)?T.similar:0.7, containment:(T&&T.containment!=null)?T.containment:0.95 };
      state.settingsOpen = false;
      try { var _st=localStorage.getItem('sms-thresholds'); if(_st){ var _o=JSON.parse(_st); ['duplicate','similar','containment'].forEach(function(k){ if(typeof _o[k]==='number'){ state.thresholds[k]=Math.max(0,Math.min(1,_o[k])); } }); } } catch(e){ }
      T = state.thresholds;
    
      function tierAt(c){ c=(c==null)?0:c; return c>=state.thresholds.duplicate?'duplicate':(c>=state.thresholds.similar?'similar':'distinct'); }
      function buildClusters(flagged){
        var parent={}, ids={};
        function find(x){ while(parent[x]!==x){ parent[x]=parent[parent[x]]; x=parent[x]; } return x; }
        function uni(a,b){ if(!(a in parent)){ parent[a]=a; } if(!(b in parent)){ parent[b]=b; } parent[find(a)]=find(b); }
        for(var i=0;i<flagged.length;i++){ var p=flagged[i]; if(p.tier==='duplicate'){ ids[p.idA]=1; ids[p.idB]=1; uni(p.idA,p.idB); } }
        var groups={};
        Object.keys(ids).forEach(function(id){ var r=find(id); (groups[r]=groups[r]||[]).push(id); });
        var out=[];
        Object.keys(groups).forEach(function(r){ var mem=groups[r]; if(mem.length<2){ return; }
          mem.sort(function(a,b){ return (((MODELS[a]||{}).name)||'').localeCompare(((MODELS[b]||{}).name)||''); });
          out.push({ size:mem.length, members:mem.map(function(id){ var m=MODELS[id]||{}; return { id:id, model:m.name||id, workspace:m.workspace||'' }; }) });
        });
        out.sort(function(a,b){ return b.size-a.size; });
        out.forEach(function(c,i){ c.id=i+1; });
        return out;
      }
      function buildMatrix(flagged){
        var idset={};
        for(var i=0;i<flagged.length;i++){ idset[flagged[i].idA]=1; idset[flagged[i].idB]=1; }
        var ids=Object.keys(idset).sort(function(a,b){ var ma=MODELS[a]||{}, mb=MODELS[b]||{}; return ((ma.name||'').localeCompare(mb.name||''))||((ma.workspace||'').localeCompare(mb.workspace||'')); });
        var pos={}, n=ids.length; for(var k=0;k<n;k++){ pos[ids[k]]=k; }
        var z=[]; for(var r=0;r<n;r++){ var row=[]; for(var c=0;c<n;c++){ row.push(r===c?1:0); } z.push(row); }
        for(var f=0;f<flagged.length;f++){ var p=flagged[f], a=pos[p.idA], b=pos[p.idB]; if(a!=null&&b!=null){ z[a][b]=z[b][a]=(p.composite||0); } }
        return { labels:ids.map(function(id){ return (MODELS[id]||{}).name||id; }), workspaces:ids.map(function(id){ return (MODELS[id]||{}).workspace||''; }), z:z };
      }
      function recompute(){
        var all=DATA.allPairs||[], flagged=[], cont=[];
        for(var i=0;i<all.length;i++){ var p=all[i]; p.tier=tierAt(p.composite); if(p.tier!=='distinct'){ flagged.push(p); } if(((p.containment==null)?0:p.containment)>=state.thresholds.containment){ cont.push(p); } }
        flagged.sort(function(a,b){ return (b.composite||0)-(a.composite||0); });
        cont.sort(function(a,b){ return (b.containment||0)-(a.containment||0); });
        DATA.pairs=flagged; DATA.containment=cont; DATA.clusters=buildClusters(flagged); DATA.matrix=buildMatrix(flagged);
        var dup=0,sim=0; for(var j=0;j<flagged.length;j++){ if(flagged[j].tier==='duplicate'){ dup++; } else { sim++; } }
        S.duplicatePairs=dup; S.similarPairs=sim; S.containmentCandidates=cont.length; S.clusters=DATA.clusters.length;
      }
    
      function esc(s){ return String(s==null?'':s).replace(/[&<>"']/g, function(c){ return {'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]; }); }
      function num(v){ return (v==null)?'&mdash;':(Math.round(v*100)/100).toFixed(2); }
      function bar(v, cls){ var w=Math.max(0,Math.min(1,v||0))*100; return '<div class="track"><div class="fill '+(cls||'')+'" style="width:'+w.toFixed(1)+'%"></div></div>'; }
    
      function buildTabs(){ return [
        {id:'overview', label:'Overview'},
        {id:'pairs', label:'Duplicates', count:DATA.pairs.length, show:DATA.pairs.length>0},
        {id:'containment', label:'Containment', count:DATA.containment.length, show:DATA.containment.length>0},
        {id:'clusters', label:'Clusters', count:DATA.clusters.length, show:DATA.clusters.length>0},
        {id:'compare', label:'Compare', show:MLIST.length>1},
        {id:'matrix', label:'Matrix', show:DATA.matrix.labels.length>1}
      ].filter(function(t){ return t.show!==false; }); }
    
      function headline(){
        if(DATA.pairs.length===0 && DATA.clusters.length===0 && DATA.containment.length===0){
          return 'No duplicates or near-duplicates found across '+S.models+' models at the current thresholds.';
        }
        var parts=[];
        if(S.duplicatePairs){ parts.push(S.duplicatePairs+' duplicate'); }
        if(S.similarPairs){ parts.push(S.similarPairs+' near-duplicate'); }
        var lead = parts.length ? parts.join(' and ')+' pair'+(((S.duplicatePairs+S.similarPairs)!==1)?'s':'') : 'overlapping models';
        var tail='';
        if(S.containmentCandidates){ tail += ', plus '+S.containmentCandidates+' subset relationship'+((S.containmentCandidates!==1)?'s':''); }
        if(S.clusters){ tail += ', forming '+S.clusters+' duplicate cluster'+((S.clusters!==1)?'s':''); }
        return 'Across '+S.models+' models, found '+lead+tail+'.';
      }
    
      function verdict(p){
        if(p.tier==='duplicate' && p.crossWorkspace){ return 'Very likely the same model in two workspaces &mdash; consolidate to a single source.'; }
        if(p.tier==='duplicate' && p.sameName){ return 'Duplicate models sharing a name &mdash; consolidate.'; }
        if(p.tier==='duplicate'){ return 'Near-identical structure &mdash; a strong consolidation candidate.'; }
        if((p.relationship==='model_a_contains_model_b'||p.relationship==='model_b_contains_model_a') && p.containment>=T.containment){ return 'One model is effectively a subset of the other &mdash; consider retiring the smaller one.'; }
        return 'Substantial overlap &mdash; review for shared logic or consolidation.';
      }
    
      function evItem(label, v){ return '<div class="ev"><div class="evl">'+label+'</div>'+bar(v,'mini')+'<div class="evv">'+num(v)+'</div></div>'; }
      function cmpBtn(a, b, label){ return '<button class="btnlink cmpbtn" data-cmp-a="'+esc(a)+'" data-cmp-b="'+esc(b)+'">'+label+' &rsaquo;</button>'; }
    
      function pairCard(p){
        var t = (p.tier==='duplicate')?'dup':'sim';
        var tl = (p.tier==='duplicate')?'Duplicate':'Similar';
        var chips = (p.crossWorkspace?'<span class="chip xws">cross-workspace</span>':'') + (p.sameName?'<span class="chip">same name</span>':'');
        return '<div class="pcard"><div class="phead"><div class="pinfo">'+
          '<div class="ptop"><span class="pill '+t+'">'+tl+'</span>'+chips+'</div>'+
          '<div class="pnames"><span class="mname">'+esc(p.modelA)+'</span> <span class="wname">&middot; '+esc(p.workspaceA)+'</span> <span class="vs">&harr;</span> <span class="mname">'+esc(p.modelB)+'</span> <span class="wname">&middot; '+esc(p.workspaceB)+'</span></div>'+
          '</div><div class="pscore">'+bar(p.composite)+'<div class="scoreval">'+num(p.composite)+' composite</div></div><div class="chev">&rsaquo;</div></div>'+
          '<div class="pdetail"><div class="verdict">'+verdict(p)+'</div><div class="evid">'+
            evItem('Tables', p.jaccard.tables)+evItem('Columns', p.jaccard.columns)+evItem('Measures', p.jaccard.measures)+
            evItem('Relationships', p.jaccard.relationships)+evItem('Data sources', p.jaccard.datasources)+evItem('DAX text', p.daxCosine)+evItem('Containment', p.containment)+
          '</div><div class="pdact">'+cmpBtn(p.idA, p.idB, 'Compare structure')+'</div></div></div>';
      }
    
      function contCard(p){
        var eqv = (p.relationship==='equivalent');
        var a,b,cov;
        if(eqv){ a=p.modelA; b=p.modelB; cov=Math.max(p.aInB||0, p.bInA||0); }
        else if((p.aInB||0) >= (p.bInA||0)){ a=p.modelA; b=p.modelB; cov=p.aInB; }
        else { a=p.modelB; b=p.modelA; cov=p.bInA; }
        var sym = eqv ? '&equiv;' : '&sub;';
        var note = eqv ? 'Equivalent &mdash; each model contains the other.' : (esc(a)+' is contained in '+esc(b)+'.');
        return '<div class="pcard"><div class="phead"><div class="pinfo">'+
          '<div class="pnames"><span class="mname">'+esc(a)+'</span> <span class="vs">'+sym+'</span> <span class="mname">'+esc(b)+'</span></div>'+
          '<div class="wname" style="margin-top:5px;">'+note+'</div></div>'+
          '<div class="pscore">'+bar(cov,'cov')+'<div class="scoreval">'+num(cov)+' coverage</div></div><div class="chev">&rsaquo;</div></div>'+
          '<div class="pdetail"><div class="evid">'+evItem('A in B', p.aInB)+evItem('B in A', p.bInA)+evItem('Composite', p.composite)+evItem('Containment', p.containment)+'</div>'+
          '<div class="pdact">'+cmpBtn(p.idA, p.idB, 'Compare structure')+'</div></div></div>';
      }
    
      function filteredPairs(){
        var q = state.pairSearch.trim().toLowerCase();
        return DATA.pairs.filter(function(p){
          if(state.pairTier!=='all' && p.tier!==state.pairTier){ return false; }
          if(!q){ return true; }
          return (p.modelA+' '+p.workspaceA+' '+p.modelB+' '+p.workspaceB).toLowerCase().indexOf(q)>=0;
        });
      }
      function filteredCont(){
        var q = state.contSearch.trim().toLowerCase();
        return DATA.containment.filter(function(p){
          if(!q){ return true; }
          return (p.modelA+' '+p.workspaceA+' '+p.modelB+' '+p.workspaceB).toLowerCase().indexOf(q)>=0;
        });
      }
    
      function overviewHTML(){
        var kpis = [
          ['accent', S.models, 'Models scanned'],
          [S.duplicatePairs?'warn':'', S.duplicatePairs, 'Duplicate pairs'],
          ['', S.similarPairs, 'Similar pairs'],
          ['', S.containmentCandidates, 'Containment'],
          ['', S.clusters, 'Clusters']
        ].map(function(c){ return '<div class="kpi '+c[0]+'"><div class="n">'+c[1]+'</div><div class="l">'+c[2]+'</div></div>'; }).join('');
        var body;
        if(DATA.pairs.length>0){
          var top = DATA.pairs.slice(0,5).map(pairCard).join('');
          var seeAll = (DATA.pairs.length>5) ? '<button class="btnlink" data-goto="pairs">See all '+DATA.pairs.length+' &rsaquo;</button>' : '';
          body = '<div class="secrow"><h3 class="sec">Top consolidation candidates</h3>'+seeAll+'</div><div class="plist">'+top+'</div>';
        } else {
          body = '<div class="empty">No duplicates or near-duplicates at the current thresholds. Open the &#9881; settings above and lower the Duplicate / Similar thresholds to widen the search.</div>';
        }
        return '<div class="head2">'+esc(headline())+'</div><div class="kpis">'+kpis+'</div>'+body;
      }
    
      function pairsHTML(){
        return '<div class="toolbar"><input class="search" type="text" placeholder="Search models or workspaces" value="'+esc(state.pairSearch)+'">'+
          '<div class="chips">'+
            '<button class="fchip'+(state.pairTier==='all'?' on':'')+'" data-tier="all">All</button>'+
            '<button class="fchip'+(state.pairTier==='duplicate'?' on':'')+'" data-tier="duplicate">Duplicates</button>'+
            '<button class="fchip'+(state.pairTier==='similar'?' on':'')+'" data-tier="similar">Similar</button>'+
          '</div></div><div class="plist"></div>';
      }
      function contHTML(){
        return '<div class="toolbar"><input class="search" type="text" placeholder="Search models or workspaces" value="'+esc(state.contSearch)+'"></div><div class="clist"></div>';
      }
      function clustersHTML(){
        if(DATA.clusters.length===0){ return '<div class="empty">No duplicate clusters were found.</div>'; }
        return '<div class="grid2">'+ DATA.clusters.map(function(c){
          var cbtn = (c.members.length>1) ? '<div class="cbtnrow">'+cmpBtn(c.members[0].id, c.members[1].id, 'Compare members')+'</div>' : '';
          return '<div class="card"><div class="crow"><div class="mname">Cluster '+c.id+'</div><span class="badge">'+c.size+' models</span></div><ul class="members">'+
            c.members.map(function(m){ return '<li><span class="mm">'+esc(m.model)+'</span><span class="w">'+esc(m.workspace)+'</span></li>'; }).join('')+
          '</ul>'+cbtn+'</div>';
        }).join('') +'</div>';
      }
      function cellColor(v){
        if(v==null){ return state.theme==='dark' ? '#242426' : '#f2f2f4'; }
        var x = Math.max(0,Math.min(1,v));
        if(state.theme==='dark'){ return 'hsl(211,70%,'+(22+x*42).toFixed(0)+'%)'; }
        return 'hsl(211,88%,'+(94-x*58).toFixed(0)+'%)';
      }
      function matrixHTML(){
        var m = DATA.matrix, n = m.labels.length;
        var h = '<div class="mwrap"><div class="mgrid" style="grid-template-columns:200px repeat('+n+',30px);">';
        h += '<div></div>';
        for(var j=0;j<n;j++){ h += '<div class="mcolh" title="'+esc(m.labels[j])+' &middot; '+esc(m.workspaces[j])+'">'+(j+1)+'</div>'; }
        for(var i=0;i<n;i++){
          h += '<div class="mrowh" title="'+esc(m.labels[i])+' &middot; '+esc(m.workspaces[i])+'"><span class="mnum">'+(i+1)+'</span>'+esc(m.labels[i])+'</div>';
          for(var k=0;k<n;k++){
            var v = m.z[i][k];
            h += '<div class="mcell" title="'+esc(m.labels[i])+' &harr; '+esc(m.labels[k])+': '+num(v)+'" style="background:'+cellColor(v)+'"></div>';
          }
        }
        h += '</div><div class="mlegend">Composite similarity<span class="mbar"></span>low to high</div></div>';
        return h;
      }
    
      function indexBy(list){ var m={}; for(var i=0;i<list.length;i++){ m[list[i].key]=list[i]; } return m; }
      function has(o,k){ return Object.prototype.hasOwnProperty.call(o,k); }
      function unionKeys(a,b){ var s={}; Object.keys(a).forEach(function(k){s[k]=1;}); Object.keys(b).forEach(function(k){s[k]=1;}); return Object.keys(s); }
      function statusOf(k,a,b){ var iA=has(a,k),iB=has(b,k); return iA&&iB?'both':(iA?'onlyA':'onlyB'); }
      function stag(st){ var t=st==='onlyA'?'A':(st==='onlyB'?'B':(st==='changed'?'chg':'=')); return '<span class="stag '+st+'">'+t+'</span>'; }
      function counts(shared,oa,ob,extra){ return '<span class="dcounts">'+shared+' shared'+(extra||'')+' &middot; <span class="cA">'+oa+' only in A</span> &middot; <span class="cB">'+ob+' only in B</span></span>'; }
      function sectionWrap(secId,title,countsHtml,body){ var col=state.cmpCollapsed[secId]?' collapsed':''; return '<div class="diffsec'+col+'"><div class="diffhead" data-sec="'+secId+'"><span class="dtitle"><span class="chevs">&rsaquo;</span>'+title+'</span>'+countsHtml+'</div><div class="diffbody">'+(body||'<div class="diffnone">Nothing to show.</div>')+'</div></div>'; }
    
      function groupCols(cols){ var g={}; for(var i=0;i<cols.length;i++){ var c=cols[i]; if(!g[c.tableKey]){ g[c.tableKey]={}; } g[c.tableKey][c.key]=c; } return g; }
    
      function tableSection(a,b){
        var ta=indexBy(a.tables), tb=indexBy(b.tables), ca=groupCols(a.columns), cb=groupCols(b.columns);
        var kset={}; unionKeys(ta,tb).forEach(function(k){kset[k]=1;}); Object.keys(ca).forEach(function(k){kset[k]=1;}); Object.keys(cb).forEach(function(k){kset[k]=1;});
        var keys=Object.keys(kset).sort();
        var shared=0,oa=0,ob=0,rows='';
        keys.forEach(function(tk){
          var inA=has(ta,tk)||has(ca,tk), inB=has(tb,tk)||has(cb,tk);
          var st=inA&&inB?'both':(inA?'onlyA':'onlyB');
          if(st==='both'){shared++;} else if(st==='onlyA'){oa++;} else {ob++;}
          var name=(ta[tk]&&ta[tk].name)||(tb[tk]&&tb[tk].name)||tk;
          var colsA=ca[tk]||{}, colsB=cb[tk]||{}, ckeys=unionKeys(colsA,colsB).sort();
          var cOA=0,cOB=0,colRows='';
          ckeys.forEach(function(ck){
            var cst=statusOf(ck,colsA,colsB);
            if(cst==='onlyA'){cOA++;} else if(cst==='onlyB'){cOB++;}
            if(state.cmpDiffOnly && cst==='both'){ return; }
            var ci=colsA[ck]||colsB[ck];
            colRows+='<div class="diffcol '+cst+'">'+stag(cst)+'<span>'+esc(ci.name)+'</span></div>';
          });
          var cdiff=cOA+cOB;
          if(state.cmpDiffOnly && st==='both' && cdiff===0){ return; }
          var meta='<span class="tmeta">'+ckeys.length+' cols'+(cdiff?(' &middot; <span class="cchg">'+cdiff+' differ</span>'):'')+'</span>';
          var openCls=(st==='both'&&cdiff>0)?' open':'';
          var cols=colRows?('<div class="cols">'+colRows+'</div>'):'';
          rows+='<div class="difftbl'+openCls+'"><div class="diffrow '+st+' thead" data-tbl><span class="chev2">&rsaquo;</span>'+stag(st)+'<span class="dtext">'+esc(name)+meta+'</span></div>'+cols+'</div>';
        });
        if(!keys.length){ return ''; }
        return sectionWrap('tables', 'Tables &amp; columns', counts(shared,oa,ob), rows);
      }
    
      function measureSection(a,b){
        var ma=indexBy(a.measures), mb=indexBy(b.measures), keys=unionKeys(ma,mb).sort();
        var shared=0,oa=0,ob=0,chg=0,rows='';
        keys.forEach(function(k){
          var iA=ma[k], iB=mb[k], st;
          if(iA&&iB){ if(iA.daxHash===iB.daxHash){ st='both'; shared++; } else { st='changed'; chg++; } }
          else if(iA){ st='onlyA'; oa++; } else { st='onlyB'; ob++; }
          if(state.cmpDiffOnly && st==='both'){ return; }
          var name=(iA&&iA.name)||(iB&&iB.name)||k, expandable=(st!=='both'), detail='';
          if(st==='changed'){ detail=daxDiff(daxText(iA),daxText(iB)); }
          else if(st==='onlyA'){ detail=daxBlock(daxText(iA),'del'); }
          else if(st==='onlyB'){ detail=daxBlock(daxText(iB),'add'); }
          var openCls=(st==='changed')?' open':'';
          rows+='<div class="difftbl'+openCls+'"><div class="diffrow '+st+' thead"'+(expandable?' data-tbl':'')+'>'+(expandable?'<span class="chev2">&rsaquo;</span>':'<span class="chev2 hide">&rsaquo;</span>')+stag(st)+'<span class="dtext">'+esc(name)+'</span></div>'+(expandable?('<div class="cols"><div class="mdax">'+detail+'</div></div>'):'')+'</div>';
        });
        if(!keys.length){ return ''; }
        return sectionWrap('measures', 'Measures', counts(shared,oa,ob,' &middot; <span class="cchg">'+chg+' changed</span>'), rows);
      }
    
      function listSection(secId,title,listA,listB,fmt){
        var ia=indexBy(listA), ib=indexBy(listB), keys=unionKeys(ia,ib).sort();
        var shared=0,oa=0,ob=0,rows='';
        keys.forEach(function(k){
          var st=statusOf(k,ia,ib);
          if(st==='both'){shared++;} else if(st==='onlyA'){oa++;} else {ob++;}
          if(state.cmpDiffOnly && st==='both'){ return; }
          rows+='<div class="diffrow '+st+'">'+stag(st)+'<span class="dtext">'+fmt(ia[k]||ib[k])+'</span></div>';
        });
        if(!keys.length){ return ''; }
        return sectionWrap(secId, title, counts(shared,oa,ob), rows);
      }
    
      function daxBlock(text,kind){ return String(text||'').split('\n').map(function(l){ return '<div class="dline '+kind+'">'+esc(l||' ')+'</div>'; }).join(''); }
      function daxDiff(aText,bText){
        var A=String(aText||'').split('\n'), B=String(bText||'').split('\n'), n=A.length, m=B.length;
        if(n>300||m>300){ return '<div class="dcap">Expressions too long to line-diff; showing both.</div><div class="dsplit"><div>'+daxBlock(aText,'del')+'</div><div>'+daxBlock(bText,'add')+'</div></div>'; }
        var dp=[]; for(var i=0;i<=n;i++){ dp.push(new Array(m+1).fill(0)); }
        for(var i=n-1;i>=0;i--){ for(var j=m-1;j>=0;j--){ dp[i][j]=(A[i]===B[j])?dp[i+1][j+1]+1:Math.max(dp[i+1][j],dp[i][j+1]); } }
        var out=[], i=0, j=0;
        while(i<n && j<m){
          if(A[i]===B[j]){ out.push('<div class="dline same">'+esc(A[i]||' ')+'</div>'); i++; j++; }
          else if(dp[i+1][j]>=dp[i][j+1]){ out.push('<div class="dline del">'+esc(A[i]||' ')+'</div>'); i++; }
          else { out.push('<div class="dline add">'+esc(B[j]||' ')+'</div>'); j++; }
        }
        while(i<n){ out.push('<div class="dline del">'+esc(A[i]||' ')+'</div>'); i++; }
        while(j<m){ out.push('<div class="dline add">'+esc(B[j]||' ')+'</div>'); j++; }
        return out.join('');
      }
    
      function selOptions(selId){ return MLIST.map(function(mm){ return '<option value="'+esc(mm.id)+'"'+(mm.id===selId?' selected':'')+'>'+esc(mm.name)+' / '+esc(mm.workspace)+'</option>'; }).join(''); }
      function compareHTML(){
        if(MLIST.length<2){ return '<div class="empty">At least two compared models are needed for a structure diff.</div>'; }
        return '<div class="cmpbar"><div class="cmppick"><select class="cmpsel" data-side="a">'+selOptions(state.cmpA)+'</select>'+
          '<button class="swap" title="Swap A and B">&#8646;</button>'+
          '<select class="cmpsel" data-side="b">'+selOptions(state.cmpB)+'</select></div>'+
          '<label class="difftoggle"><input type="checkbox" class="cmponly"'+(state.cmpDiffOnly?' checked':'')+'> Show only differences</label></div>'+
          '<div class="cmpbody"></div>';
      }
      function updateCompare(){
        var body=root.querySelector('.cmpbody'); if(!body){ return; }
        var a=MODELS[state.cmpA], b=MODELS[state.cmpB];
        if(!a||!b){ body.innerHTML='<div class="empty">Select two models to compare.</div>'; return; }
        if(state.cmpA===state.cmpB){ body.innerHTML='<div class="empty">Choose two different models.</div>'; return; }
        var head='<div class="cmphead"><div class="cmpm"><span class="dot dotA"></span>'+esc(a.name)+' <span class="wname">/ '+esc(a.workspace)+'</span></div><div class="cmpvs">vs</div><div class="cmpm"><span class="dot dotB"></span>'+esc(b.name)+' <span class="wname">/ '+esc(b.workspace)+'</span></div></div>';
        var out=head+tableSection(a,b)+measureSection(a,b)+
          listSection('relationships', 'Relationships', a.relationships, b.relationships, function(x){ return esc(x.from)+' &rarr; '+esc(x.to); })+
          listSection('datasources', 'Data sources', a.datasources, b.datasources, function(x){ return esc(x.name); });
        body.innerHTML=out;
      }
      function wireCompare(v){
        v.querySelectorAll('.cmpsel').forEach(function(sel){ sel.addEventListener('change', function(){ if(sel.dataset.side==='a'){ state.cmpA=sel.value; } else { state.cmpB=sel.value; } updateCompare(); }); });
        var sw=v.querySelector('.swap'); if(sw){ sw.addEventListener('click', function(){ var t=state.cmpA; state.cmpA=state.cmpB; state.cmpB=t; v.innerHTML=compareHTML(); wireCompare(v); updateCompare(); }); }
        var only=v.querySelector('.cmponly'); if(only){ only.addEventListener('change', function(){ state.cmpDiffOnly=only.checked; updateCompare(); }); }
      }
      function openCompare(a,b){ if(a){ state.cmpA=String(a); } if(b){ state.cmpB=String(b); } setTab('compare'); }
    
      function settingsHTML(){
        var rows=[['duplicate','Duplicate','composite at/above = duplicate'],['similar','Similar','composite at/above = near-duplicate'],['containment','Containment','coverage at/above = subset candidate']].map(function(r){ var v=state.thresholds[r[0]]; return '<div class="setrow"><div class="setlab">'+r[1]+'<span class="sethint">'+r[2]+'</span></div><input type="range" class="setrange" data-k="'+r[0]+'" min="0" max="1" step="0.01" value="'+v+'"><input type="number" class="setnum" data-k="'+r[0]+'" min="0" max="1" step="0.01" value="'+v.toFixed(2)+'"></div>'; }).join('');
        return '<div class="settings'+(state.settingsOpen?' open':'')+'"><div class="setinner">'+rows+'<div class="setfoot"><span class="setnote">Adjust the tiers live &mdash; nothing is re-scored. Weights &amp; blocking live in the compute notebook.</span><button class="btnlink setreset" data-setreset>Reset to defaults</button></div></div></div>';
      }
      function saveThresholds(){ try { localStorage.setItem('sms-thresholds', JSON.stringify(state.thresholds)); } catch(e){ } }
      function refreshTabsBar(){ var bar=root.querySelector('.tabs'); if(bar){ bar.outerHTML=tabsHTML(); } root.querySelectorAll('.tab').forEach(function(b){ b.addEventListener('click', function(){ setTab(b.dataset.tab); }); }); }
      function refreshFoot(){ var f=root.querySelector('.foot'); if(f){ f.outerHTML=footHTML(); } }
      function applyThresholds(){ recompute(); if(!buildTabs().some(function(t){ return t.id===state.tab; })){ state.tab='overview'; } refreshTabsBar(); renderView(); refreshFoot(); }
      function wireSettings(){
        var g=root.querySelector('[data-gear]');
        if(g){ g.addEventListener('click', function(){ state.settingsOpen=!state.settingsOpen; var pane=root.querySelector('.settings'); if(pane){ pane.classList.toggle('open', state.settingsOpen); } g.classList.toggle('on', state.settingsOpen); }); }
        root.querySelectorAll('.setrange, .setnum').forEach(function(inp){ inp.addEventListener('input', function(){ var k=inp.dataset.k, val=parseFloat(inp.value); if(isNaN(val)){ return; } val=Math.max(0,Math.min(1,val)); state.thresholds[k]=val; root.querySelectorAll('.setrange[data-k="'+k+'"]').forEach(function(x){ if(x!==inp){ x.value=val; } }); root.querySelectorAll('.setnum[data-k="'+k+'"]').forEach(function(x){ if(x!==inp){ x.value=val.toFixed(2); } }); saveThresholds(); applyThresholds(); }); });
        var rs=root.querySelector('[data-setreset]');
        if(rs){ rs.addEventListener('click', function(){ state.thresholds={ duplicate:DATA.thresholds.duplicate, similar:DATA.thresholds.similar, containment:DATA.thresholds.containment }; saveThresholds(); render(); }); }
      }
      function footHTML(){
        return '<div class="foot">Duplicate &ge; '+T.duplicate+' composite &middot; Similar &ge; '+T.similar+' &middot; Containment &ge; '+T.containment+' coverage. Metadata-based (tables, columns, measures &amp; DAX, relationships, data sources) &mdash; confirm intent before consolidating.</div>';
      }
      function tabsHTML(){
        return '<div class="tabs">'+ buildTabs().map(function(t){
          return '<button class="tab'+(t.id===state.tab?' active':'')+'" data-tab="'+t.id+'">'+t.label+(t.count!=null?' <span class="tcount">'+t.count+'</span>':'')+'</button>';
        }).join('') +'</div>';
      }
      function headerHTML(){
        return '<div class="apphead"><div><div class="eyebrow">Semantic Model Similarity</div><div class="apptitle">Duplicate &amp; overlap explorer</div></div>'+
          '<div class="hright"><div class="hctrls"><button class="gear'+(state.settingsOpen?' on':'')+'" data-gear title="Adjust thresholds">&#9881;</button><div class="themeswitch"><button class="tsw" data-theme-set="light" title="Light">&#9728;</button><button class="tsw" data-theme-set="dark" title="Dark">&#9789;</button></div></div>'+
          '<div class="gen">'+esc(DATA.generatedAt)+'</div></div></div>';
      }
    
      function updatePairList(){
        var list=root.querySelector('.plist'); if(!list){ return; }
        var items=filteredPairs();
        list.innerHTML=items.length?items.map(pairCard).join(''):'<div class="empty">No pairs match your search.</div>';
      }
      function updateContList(){
        var list=root.querySelector('.clist'); if(!list){ return; }
        var items=filteredCont();
        list.innerHTML=items.length?items.map(contCard).join(''):'<div class="empty">No containment candidates match your search.</div>';
      }
    
      function renderView(){
        var v=root.querySelector('#sms-view'); if(!v){ return; }
        if(state.tab==='overview'){ v.innerHTML=overviewHTML(); }
        else if(state.tab==='pairs'){ v.innerHTML=pairsHTML(); updatePairList(); wirePairs(v); }
        else if(state.tab==='containment'){ v.innerHTML=contHTML(); updateContList(); wireCont(v); }
        else if(state.tab==='clusters'){ v.innerHTML=clustersHTML(); }
        else if(state.tab==='compare'){ v.innerHTML=compareHTML(); wireCompare(v); updateCompare(); }
        else if(state.tab==='matrix'){ v.innerHTML=matrixHTML(); }
      }
      function wirePairs(v){
        var s=v.querySelector('.search');
        if(s){ s.addEventListener('input', function(e){ state.pairSearch=e.target.value; updatePairList(); }); }
        v.querySelectorAll('.fchip').forEach(function(b){ b.addEventListener('click', function(){ state.pairTier=b.dataset.tier; v.querySelectorAll('.fchip').forEach(function(x){ x.classList.toggle('on', x===b); }); updatePairList(); }); });
      }
      function wireCont(v){
        var s=v.querySelector('.search');
        if(s){ s.addEventListener('input', function(e){ state.contSearch=e.target.value; updateContList(); }); }
      }
      function setTab(id){ state.tab=id; root.querySelectorAll('.tab').forEach(function(b){ b.classList.toggle('active', b.dataset.tab===id); }); renderView(); }
      function applyTheme(){
        root.setAttribute('data-theme', state.theme);
        root.querySelectorAll('.tsw').forEach(function(b){ b.classList.toggle('on', b.dataset.themeSet===state.theme); });
        try { localStorage.setItem('sms-theme', state.theme); } catch(e){ }
        if(state.tab==='matrix'){ renderView(); }
      }
    
      function render(){
        recompute();
        root.setAttribute('data-theme', state.theme);
        root.innerHTML=headerHTML()+settingsHTML()+tabsHTML()+'<div id="sms-view"></div>'+footHTML();
        root.querySelectorAll('.tab').forEach(function(b){ b.addEventListener('click', function(){ setTab(b.dataset.tab); }); });
        root.querySelectorAll('.tsw').forEach(function(b){ b.addEventListener('click', function(){ state.theme=b.dataset.themeSet; applyTheme(); }); });
        wireSettings();
        var view=root.querySelector('#sms-view');
        view.addEventListener('click', function(e){
          var goto=e.target.closest('[data-goto]');
          if(goto){ setTab(goto.dataset.goto); return; }
          var cmp=e.target.closest('[data-cmp-a]');
          if(cmp){ openCompare(cmp.dataset.cmpA, cmp.dataset.cmpB); return; }
          var sec=e.target.closest('[data-sec]');
          if(sec){ var sid=sec.dataset.sec; state.cmpCollapsed[sid]=!state.cmpCollapsed[sid]; sec.parentElement.classList.toggle('collapsed'); return; }
          var tbl=e.target.closest('[data-tbl]');
          if(tbl){ tbl.parentElement.classList.toggle('open'); return; }
          var head=e.target.closest('.phead');
          if(head){ head.parentElement.classList.toggle('open'); }
        });
        applyTheme();
        renderView();
      }
    
      render();
    })();
    </script>"""
    
    _app_json = json.dumps(_app_data).replace("<", "\\u003c")
    displayHTML(_APP_TEMPLATE.replace("__APP_DATA__", _app_json))


In [ ]:
render_results()